# Feature Engineering

In [1]:
import pandas as pd

In [2]:
train = pd.read_csv("../data/raw/train.csv")

C:\Users\ASUS TUF\AppData\Local\Temp\ipykernel_21104\513577916.py:1: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv("../data/raw/train.csv")


## 4.1 Date Features

In [3]:
train["Date"] = pd.to_datetime(train["Date"])

In [4]:
train["Year"] = train["Date"].dt.year

In [5]:
train["Month"] = train["Date"].dt.month

In [6]:
train["Day"] = train["Date"].dt.day

In [7]:
train[["Date", "Year", "Month", "Day"]].head()

,Date,Year,Month,Day
0,2015-07-31,2015,7,31
1,2015-07-31,2015,7,31
2,2015-07-31,2015,7,31
3,2015-07-31,2015,7,31
4,2015-07-31,2015,7,31


### Task Completed
Dateમાંથી Year, Month અને Day features બનાવ્યા.

## 4.2 Week / Month / Year Features

In [8]:
train["WeekOfYear"] = train["Date"].dt.isocalendar().week

In [9]:
train[["Date", "Year", "Month", "WeekOfYear"]].head()

,Date,Year,Month,WeekOfYear
0,2015-07-31,2015,7,31
1,2015-07-31,2015,7,31
2,2015-07-31,2015,7,31
3,2015-07-31,2015,7,31
4,2015-07-31,2015,7,31


### Task Completed
WeekOfYear feature બનાવ્યું.

## 4.3 Store-related Features

In [10]:
store = pd.read_csv("../data/raw/store.csv")

In [11]:
store.shape

(1115, 10)

In [12]:
store.columns

Index(['Store', 'StoreType', 'Assortment', 'CompetitionDistance',
       'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2',
       'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval'],
      dtype='str')

In [13]:
store["Store"].nunique()

1115

In [14]:
store["Store"].duplicated().sum()

np.int64(0)

In [15]:
train = train.merge(store, on="Store", how="left")

In [16]:
train.shape

(1017209, 22)

In [17]:
train.columns

Index(['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo',
       'StateHoliday', 'SchoolHoliday', 'Year', 'Month', 'Day', 'WeekOfYear',
       'StoreType', 'Assortment', 'CompetitionDistance',
       'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2',
       'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval'],
      dtype='str')

In [18]:
train[["Store", "StoreType", "Assortment", "CompetitionDistance", "Promo2"]].head()

,Store,StoreType,Assortment,CompetitionDistance,Promo2
0,1,c,a,1270.0,0
1,2,a,a,570.0,1
2,3,a,a,14130.0,1
3,4,c,c,620.0,0
4,5,a,a,29910.0,0


### Task Completed
Storeની additional information train data સાથે merge કરી.

## 4.4 Holiday & Promo Features

In [26]:
train["StateHoliday"].value_counts()

StateHoliday
0    855087
0    131072
a     20260
b      6690
c      4100
Name: count, dtype: int64

In [27]:
train["StateHoliday"] = train["StateHoliday"].astype(str)

In [28]:
train["StateHoliday"].value_counts()

StateHoliday
0    986159
a     20260
b      6690
c      4100
Name: count, dtype: int64

In [29]:
train["IsStateHoliday"] = (train["StateHoliday"] != "0").astype(int)

In [30]:
train["IsStateHoliday"].value_counts()

IsStateHoliday
0    986159
1     31050
Name: count, dtype: int64

In [31]:
train["SchoolHoliday"].head()

0    1
1    1
2    1
3    1
4    1
Name: SchoolHoliday, dtype: int64

### Task Completed
StateHoliday અને SchoolHoliday features તૈયાર કર્યા અને Promo પહેલેથી binary feature હોવાથી તેને નવી columnમાં બદલવાની જરૂર નથી.

## 4.5 – Lag Features

In [32]:
train = train.sort_values(["Store", "Date"])

In [34]:
train["Lag_1_Sales"] = train.groupby("Store")["Sales"].shift(1)

In [35]:
train[["Store", "Date", "Sales", "Lag_1_Sales"]].head()

,Store,Date,Sales,Lag_1_Sales
1016095,1,2013-01-01,0,NaN
1014980,1,2013-01-02,5530,0.0
1013865,1,2013-01-03,4327,5530.0
1012750,1,2013-01-04,4486,4327.0
1011635,1,2013-01-05,4997,4486.0


### Task Completed
દરેક Store માટે previous day's Sales પરથી `Lag_1_Sales` feature બનાવ્યું.

In [36]:
train["Lag_7_Sales"] = train.groupby("Store")["Sales"].shift(7)

In [38]:
train[["Store", "Date", "Sales", "Lag_1_Sales","Lag_7_Sales" ]].head(10)

,Store,Date,Sales,Lag_1_Sales,Lag_7_Sales
1016095,1,2013-01-01,0,NaN,NaN
1014980,1,2013-01-02,5530,0.0,NaN
1013865,1,2013-01-03,4327,5530.0,NaN
1012750,1,2013-01-04,4486,4327.0,NaN
1011635,1,2013-01-05,4997,4486.0,NaN
1010520,1,2013-01-06,0,4997.0,NaN
1009405,1,2013-01-07,7176,0.0,NaN
1008290,1,2013-01-08,5580,7176.0,0.0
1007175,1,2013-01-09,5471,5580.0,5530.0
1006060,1,2013-01-10,4892,5471.0,4327.0


### Task Completed
દરેક Store માટે 7 દિવસ પહેલાંની Sales પરથી `Lag_7_Sales` feature બનાવ્યું.

In [39]:
train["Lag_14_Sales"] = train.groupby("Store")["Sales"].shift(14)

In [40]:
train[["Store", "Date", "Sales", "Lag_1_Sales", "Lag_7_Sales", "Lag_14_Sales"]].head(20)

,Store,Date,Sales,Lag_1_Sales,Lag_7_Sales,Lag_14_Sales
1016095,1,2013-01-01,0,NaN,NaN,NaN
1014980,1,2013-01-02,5530,0.0,NaN,NaN
1013865,1,2013-01-03,4327,5530.0,NaN,NaN
1012750,1,2013-01-04,4486,4327.0,NaN,NaN
1011635,1,2013-01-05,4997,4486.0,NaN,NaN
1010520,1,2013-01-06,0,4997.0,NaN,NaN
1009405,1,2013-01-07,7176,0.0,NaN,NaN
1008290,1,2013-01-08,5580,7176.0,0.0,NaN
1007175,1,2013-01-09,5471,5580.0,5530.0,NaN
1006060,1,2013-01-10,4892,5471.0,4327.0,NaN


### Task Completed
દરેક Store માટે 14 દિવસ પહેલાંની Sales પરથી `Lag_14_Sales` feature બનાવ્યું.